# Serve NLS Model from Colab

This notebook runs the `adsabs/scix-nls-translator` model as an OpenAI-compatible
inference endpoint, exposed via ngrok so nectar can reach it.

**Requirements:**
- Google Colab with T4 GPU runtime (free tier works)
- ngrok auth token (free at https://ngrok.com)

**Output:**
- A public URL you can set as `NL_SEARCH_VLLM_ENDPOINT` in nectar's `.env.local`

## 1. Check GPU & Install Dependencies

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
!pip install -q torch transformers accelerate fastapi uvicorn pyngrok

## 2. Load Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "adsabs/scix-nls-translator"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME} on {DEVICE}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map=DEVICE,
    trust_remote_code=True,
)

print(f"Model loaded on {DEVICE}")

## 3. Quick Sanity Check

In [ ]:
messages = [
    {"role": "system", "content": 'Convert natural language to ADS search query. Output JSON: {"query": "..."}'},
    {"role": "user", "content": "Query: papers about exoplanets by Smith\nDate: 2026-02-26"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Test output: {response}")

## 4. Start FastAPI Server

In [ ]:
import json
import time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="NLS Colab Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class ChatMessage(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    model: str = "llm"
    messages: list[ChatMessage]
    max_tokens: int = 256


def generate_query(messages, max_tokens=256):
    message_dicts = [{"role": m.role, "content": m.content} for m in messages]
    prompt = tokenizer.apply_chat_template(message_dicts, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_tokens = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][prompt_tokens:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # Strip thinking tags
    if "<think>" in response:
        parts = response.split("</think>")
        if len(parts) > 1:
            response = parts[-1].strip()

    # Extract JSON query
    try:
        json_start = response.find("{")
        json_end = response.rfind("}") + 1
        if json_start >= 0 and json_end > json_start:
            data = json.loads(response[json_start:json_end])
            response = data.get("query", response)
    except json.JSONDecodeError:
        pass

    return response, prompt_tokens, len(generated_ids)


@app.get("/health")
async def health():
    return {"status": "healthy", "model": MODEL_NAME, "device": DEVICE}


@app.get("/v1/models")
async def list_models():
    return {"object": "list", "data": [{"id": "llm", "object": "model", "created": int(time.time()), "owned_by": "adsabs"}]}


@app.post("/v1/chat/completions")
async def chat_completions(request: ChatRequest):
    try:
        start = time.time()
        text, prompt_tok, comp_tok = generate_query(request.messages, request.max_tokens)
        elapsed = (time.time() - start) * 1000
        print(f"[{elapsed:.0f}ms] {text[:100]}")

        return {
            "id": f"chatcmpl-{int(time.time())}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": request.model,
            "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
            "usage": {"prompt_tokens": prompt_tok, "completion_tokens": comp_tok, "total_tokens": prompt_tok + comp_tok},
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print("FastAPI app defined")

## 5. Expose via ngrok

Get a free auth token at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import getpass

NGROK_TOKEN = getpass.getpass("Enter your ngrok auth token: ")

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading

nest_asyncio.apply()

# Set up ngrok tunnel
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("=" * 60)
print(f"Public URL: {public_url}")
print(f"")
print(f"Add to nectar .env.local:")
print(f"NL_SEARCH_VLLM_ENDPOINT={public_url}/v1/chat/completions")
print("=" * 60)

# Run server (blocks this cell)
uvicorn.run(app, host="0.0.0.0", port=8000)

## Usage

Once running, copy the `NL_SEARCH_VLLM_ENDPOINT` line above into your nectar `.env.local`, then:

```bash
cd ~/ads-dev/nectar
pnpm dev
```

Visit `http://localhost:8000/?nl_search=1` to enable the feature.

**Notes:**
- Free Colab sessions last ~12 hours before disconnecting
- The ngrok URL changes each time you restart
- Inference takes ~200-500ms per query on T4